In [233]:
import os
import pandas as pd
from datetime import datetime
import importlib
import numpy as np

np.set_printoptions(suppress=True)

In [234]:
# Relative imports
os.chdir("..")
import hidden_state_model.processor
importlib.reload(hidden_state_model.processor)
Processor = hidden_state_model.processor.Processor
os.chdir("hidden_state_model")

### Read (and compact) dataframes

In [235]:
compact = False

In [236]:
# Iterate over files in dfs/*.parquet and combine to one df
dfs = []

read = []
for file in os.listdir("data"):
    if file.endswith(".parquet"):
        read.append(file)
        print(f"Reading {file}")
        df = pd.read_parquet(f"data/{file}")
        dfs.append(df)
    if file.endswith(".csv"):
        read.append(file)
        df = pd.read_csv(f"data/{file}", index_col=0)
        dfs.append(df)

raw_df = pd.concat(dfs)

if compact and len(dfs) > 1:
    print("Compacintg dfs")
    # Move read files to trash and write combined df to dfs/combined_{timestamp}.parquet
    trash = "data/trash"
    for f in read:
        os.rename(f"data/{f}", f"{trash}/{f}")

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    raw_df.to_parquet(f"data/compacted_{timestamp}.parquet")

dfs = []  # Clear memory
raw_df

Reading 2024-08-31_00-36-28.parquet
Reading 2024-08-30_23-46-12.parquet
Reading 2024-08-31_00-40-38.parquet
Reading 2024-08-31_00-51-48.parquet
Reading 2024-08-31_00-27-50.parquet
Reading 2024-08-31_00-30-45.parquet
Reading 2024-08-31_00-41-46.parquet
Reading 2024-08-31_01-54-54.parquet
Reading 2024-08-30_22-21-59.parquet
Reading 2024-08-31_00-32-34.parquet
Reading 2024-08-31_01-01-00.parquet
Reading 2024-08-31_01-12-13.parquet
Reading 2024-08-31_00-25-13.parquet
Reading 2024-08-31_00-45-09.parquet
Reading 2024-08-31_01-16-36.parquet
Reading 2024-08-31_00-24-14.parquet
Reading 2024-08-30_23-08-29.parquet
Reading 2024-08-31_02-16-35.parquet
Reading 2024-08-30_23-34-09.parquet
Reading 2024-08-31_02-01-07.parquet
Reading 2024-08-31_01-04-17.parquet
Reading 2024-08-31_00-39-37.parquet
Reading 2024-08-31_02-02-35.parquet
Reading 2024-08-31_01-45-07.parquet


,prev_entry,public_cards,player_piles,current_player_i,bet_in_stage,bet_in_game,player_has_played,player_is_folded,first_better_i,big_blind,player_name,player_type,action,amount,p,relative_ev,rank,tiebreakers
state_id,,,,,,,,,,,,,,,,,,
4896996752,NaN,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,call,2,0.5130,0.015390,0,"[8, 4, 0, 0, 0]"
6158719728,4.896997e+09,"[41, 30, 42]","[96, 96]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.6077,0.024308,1,"[4, 8, 3, 2, 0]"
6158687296,6.158720e+09,"[41, 30, 42, 6]","[92, 92]",0,"[0, 0]","[8, 8]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,8,0.5770,0.046160,1,"[4, 8, 6, 3, 0]"
6158493008,6.158687e+09,"[41, 30, 42, 6, 28]","[84, 84]",0,"[0, 0]","[16, 16]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,check,0,0.5421,0.086736,2,"[4, 2, 8, 0, 0]"
6158678432,6.158493e+09,"[41, 30, 42, 6, 28]","[84, 78]",0,"[0, 6]","[16, 22]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,6,0.5421,0.102999,2,"[4, 2, 8, 0, 0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,6.127529e+09,"[8, 9, 36]","[152, 40]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.6240,0.024960,1,"[8, 10, 9, 4, 0]"
6431104400,6.431249e+09,"[8, 9, 36, 6]","[148, 36]",0,"[0, 0]","[8, 8]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.5459,0.043672,1,"[8, 10, 9, 6, 0]"
6127537840,6.431104e+09,"[8, 9, 36, 6]","[144, 26]",0,"[4, 10]","[12, 18]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,6,0.5459,0.081885,1,"[8, 10, 9, 6, 0]"


In [237]:
raw_df.dtypes

prev_entry           float64
public_cards          object
player_piles          object
current_player_i       int64
bet_in_stage          object
bet_in_game           object
player_has_played     object
player_is_folded      object
first_better_i         int64
big_blind              int64
player_name           object
player_type           object
action                object
amount                 int64
p                    float64
relative_ev          float64
rank                   int64
tiebreakers           object
dtype: object

## Process data

In [238]:
df = Processor(raw_df).get_processed_df()
df

,game_id,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,...,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,excess_rank,p,relative_ev,stage,player_name
4896996752,e9cd67b4-d90b-47f5-9444-28a7368cd043,0,0,0,0,0,0,0,0,0,...,0,0,0,call,2,0,0.5130,0.015390,preflop,Tord
6158719728,e9cd67b4-d90b-47f5-9444-28a7368cd043,0,0,0,0,0,2,0,0,0,...,0,0,0,raise,4,1,0.6077,0.024308,flop,Tord
6158687296,e9cd67b4-d90b-47f5-9444-28a7368cd043,0,4,0,0,0,2,0,0,0,...,0,0,0,raise,8,1,0.5770,0.046160,turn,Tord
6158493008,e9cd67b4-d90b-47f5-9444-28a7368cd043,0,4,8,0,0,2,0,0,0,...,0,0,0,check,0,1,0.5421,0.086736,river,Tord
6158678432,e9cd67b4-d90b-47f5-9444-28a7368cd043,0,4,8,0,0,2,0,0,0,...,0,0,0,call,6,1,0.5421,0.102999,river,Tord
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,a33576f9-e508-4a5d-8084-88efc86a8ece,0,0,0,0,0,2,0,0,0,...,0,0,0,raise,4,1,0.6240,0.024960,flop,Tord
6431104400,a33576f9-e508-4a5d-8084-88efc86a8ece,0,4,0,0,0,2,0,0,0,...,0,0,0,raise,4,1,0.5459,0.043672,turn,Tord
6127537840,a33576f9-e508-4a5d-8084-88efc86a8ece,0,4,4,0,0,2,0,0,0,...,0,0,0,call,6,1,0.5459,0.081885,turn,Tord
6431250896,a33576f9-e508-4a5d-8084-88efc86a8ece,0,4,4,0,0,2,0,6,0,...,0,1,0,check,0,0,0.9412,0.169416,river,Tord


In [239]:
df.dtypes

game_id                     object
raise_preflop                int64
raise_flop                   int64
raise_turn                   int64
raise_river                  int64
raise_showdown               int64
call_preflop                 int64
call_flop                    int64
call_turn                    int64
call_river                   int64
call_showdown                int64
check_preflop                int64
check_flop                   int64
check_turn                   int64
check_river                  int64
check_showdown               int64
opponent_raise_preflop       int64
opponent_raise_flop          int64
opponent_raise_turn          int64
opponent_raise_river         int64
opponent_raise_showdown      int64
opponent_call_preflop        int64
opponent_call_flop           int64
opponent_call_turn           int64
opponent_call_river          int64
opponent_call_showdown       int64
opponent_check_preflop       int64
opponent_check_flop          int64
opponent_check_turn 

## Training

In [240]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [241]:
X = df.drop(["excess_rank", "game_id", "p", "relative_ev"], axis=1)
y = df["excess_rank"]
groups = df["game_id"]  # Group by 'game_id' to ensure no data leakage

In [242]:
X

,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,call_showdown,...,opponent_call_showdown,opponent_check_preflop,opponent_check_flop,opponent_check_turn,opponent_check_river,opponent_check_showdown,action,amount,stage,player_name
4896996752,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,call,2,preflop,Tord
6158719728,0,0,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,raise,4,flop,Tord
6158687296,0,4,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,raise,8,turn,Tord
6158493008,0,4,8,0,0,2,0,0,0,0,...,0,0,1,0,0,0,check,0,river,Tord
6158678432,0,4,8,0,0,2,0,0,0,0,...,0,0,1,0,0,0,call,6,river,Tord
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,0,0,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,raise,4,flop,Tord
6431104400,0,4,0,0,0,2,0,0,0,0,...,0,0,1,0,0,0,raise,4,turn,Tord
6127537840,0,4,4,0,0,2,0,0,0,0,...,0,0,1,0,0,0,call,6,turn,Tord
6431250896,0,4,4,0,0,2,0,6,0,0,...,0,0,1,0,1,0,check,0,river,Tord


In [243]:
y

4896996752    0
6158719728    1
6158687296    1
6158493008    1
6158678432    1
             ..
6431248928    1
6431104400    1
6127537840    1
6431250896    0
6431104448    0
Name: excess_rank, Length: 1053, dtype: int64

In [244]:
# Identify categorical columns (excluding 'game_id')
categorical_cols = ["action", "stage", "player_name"]

# Preprocessing pipeline: OneHotEncoding for categorical and scaling for numerical
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(drop="first"), categorical_cols)],
    remainder="passthrough",
)

# Create the full pipeline with logistic regression
model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "classifier",
            LogisticRegression(
                multi_class="multinomial", solver="lbfgs", max_iter=1000
            ),
        ),
    ]
)

In [245]:
# Grouped train-test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (838, 34)
Test shape: (215, 34)


In [246]:
# Train the model
model.fit(X_train, y_train)

# Evaluate the model
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy}")

Accuracy: 0.8186046511627907


In [247]:
probabilities = model.predict_proba(X_test)
prob_df = pd.DataFrame(probabilities, columns=model.classes_)
prob_df["true"] = y_test.values
prob_df["pred"] = model.predict(X_test)
prob_df["correct"] = prob_df["true"] == prob_df["pred"]
prob_df["goodness"] = prob_df.apply(lambda x: x.get(x["true"]) or 0, axis=1)
print("Accuracy", prob_df["correct"].mean())
print("Mean goodness: ", prob_df["goodness"].mean())
prob_df

Accuracy 0.8186046511627907
Mean goodness:  0.7393350143051394


,0,1,2,3,4,5,true,pred,correct,goodness
0,0.969910,0.027653,3.915313e-04,0.001177,3.662191e-04,5.017956e-04,0,0,True,0.969910
1,0.999769,0.000150,2.193730e-09,0.000081,3.327146e-07,4.187887e-08,0,0,True,0.999769
2,0.993141,0.005870,6.673092e-05,0.000466,3.111897e-04,1.458366e-04,0,0,True,0.993141
3,0.197296,0.701382,7.395907e-02,0.011808,5.909114e-03,9.646461e-03,1,1,True,0.701382
4,0.124947,0.785703,4.906464e-02,0.030543,3.891976e-05,9.703344e-03,1,1,True,0.785703
...,...,...,...,...,...,...,...,...,...,...
210,0.640118,0.179964,2.147054e-05,0.178777,9.241409e-04,1.956204e-04,1,0,False,0.179964
211,0.989504,0.008844,1.653798e-04,0.000598,4.489406e-04,4.402446e-04,0,0,True,0.989504
212,0.144382,0.716254,1.066394e-01,0.010592,5.081136e-03,1.705172e-02,1,1,True,0.716254
213,0.954383,0.041150,9.583108e-04,0.001491,5.217821e-04,1.496025e-03,0,0,True,0.954383


In [248]:
# Look at incorrect predictions
print(prob_df[prob_df["correct"] == False].shape[0], "incorrect predictions:")
prob_df[prob_df["correct"] == False]

39 incorrect predictions:


,0,1,2,3,4,5,true,pred,correct,goodness
16,0.075166,0.712862,8.624678e-03,3.936329e-03,0.099197,0.100214,0,1,False,0.075166
22,0.713551,0.095111,1.322426e-03,3.659189e-02,0.067017,0.086407,1,0,False,0.095111
24,0.905885,0.077126,2.272756e-03,6.094456e-03,0.005014,0.003608,1,0,False,0.077126
57,0.993141,0.005870,6.673092e-05,4.658932e-04,0.000311,0.000146,1,0,False,0.005870
58,0.905885,0.077126,2.272756e-03,6.094456e-03,0.005014,0.003608,1,0,False,0.077126
59,0.900854,0.071063,1.756506e-03,3.488805e-03,0.009939,0.012898,1,0,False,0.071063
69,0.190412,0.794717,1.886030e-04,8.210490e-03,0.004117,0.002355,0,1,False,0.190412
70,0.080550,0.913466,1.586134e-03,1.161941e-03,0.000769,0.002467,0,1,False,0.080550
82,0.144382,0.716254,1.066394e-01,1.059226e-02,0.005081,0.017052,2,1,False,0.106639
91,0.527568,0.404370,5.857650e-03,5.119430e-02,0.003924,0.007086,2,0,False,0.005858
